# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question.** Which content pages should a reviewer look at first because their search visibility is likely to fall over the next 30 days, and does a learned model rank them better than a transparent rule?

**Decision supported.** Each review cycle a content reviewer has time for roughly 20 to 50 pages out of thousands. The ranked queue tells them where to look first and why (reason codes), and which kind of action fits: refresh or improve, protect, merge or prune, or monitor.

**Frame.** For content reviewers deciding which pages to review first, we build a ranked review queue from the FlyRank ML Internship warehouse (Google Search Console and Analytics daily performance plus content metadata). The queue scores the probability that a page's impressions persistently fall, relative to its client, over the next 30 days, and is judged by precision@K against a transparent rule baseline on a sealed test month. A wrong call costs reviewer time (a false alarm) or a longer-undetected decline (a miss). A plain rule is not enough because decline depends on many tangled, shifting signals, and a flag alone cannot order a very large flagged set. We claim only observed, directional, and decision-support results: no causal effect of any refresh, and nothing about how Google ranks pages.

In [1]:
K_LIST = [20, 50]  # review capacities the queue is judged at
print('Judged at K =', K_LIST)

Judged at K = [20, 50]


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** The full pseudonymized warehouse release of the FlyRank ML Internship dataset (hosted on Hugging Face, access-gated). No client names, domains, URLs, queries, or raw exports appear in this work.

**Tables used**
- `fact_content_daily_performance`: one row per content item per day; 78,835,655 rows from 2025-01-27 to 2026-06-30. Exact duplicate rows (6,390 groups, 0.016%) are removed with `SELECT DISTINCT`.
- `dim_content`: 519,606 content items (static attributes, creation date, status).
- `dim_clients`: client history start dates, used for eligibility.
- `fact_content_query_90d`: a single window (2026-04-02 to 2026-06-30) that overlaps the label windows, so it is not a model feature.

**Date windows.** For each month-end snapshot t, features come from t-90d to t and the label from t to t+30d. Train: t = 2025-04-30 to 2026-03-31. Validation: t = 2026-04-30. Sealed test: t = 2026-05-31 (label window June 2026). t = 2026-06-30 is used only for the final ranked list.

**Modeling universe.** Published, not-deleted pages at least 90 days old, from clients with at least 90 days of history at t, with at least 100 impressions in the 30 days before t. Pages below the floor are reported as uncovered and not labeled (about 43% of active pages in 2026-03).

**Excluded, and why.** Deleted or unpublished pages (they carried negligible traffic in 2026-03; 411,540 of 519,606 pages are kept). `content_updated_date` (82.5% of dated pages show a May to Jul 2026 update, which looks like a bulk touch). `last_optimized_date` as a feature (no training-period coverage; used only as a flag). Export-time status fields, AI-referral sessions (sparse), and GA4 metrics unless explicitly available. Product flags (health score, priority score, action type) are not in the data and are never used.

Full contract, with the query behind every number: `w03_data_contract.ipynb`.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Design constants used by every later section
FLOOR = 100          # min impressions in the 30 days before t
FEATURE_DAYS = 90
LABEL_DAYS = 30
SNAPSHOTS = {
    'train': ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
              '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31'],
    'validation': '2026-04-30',
    'test': '2026-05-31',
    'final_list': '2026-06-30',
}

# Cheap checks that the numbers in the text above still hold (full queries: w03_data_contract.ipynb)
con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(*) AS n_rows FROM {TABLES['fact_daily']}").show()
con.sql(f"SELECT is_published, is_deleted, COUNT(*) AS n FROM {TABLES['dim_content']} GROUP BY 1,2 ORDER BY n DESC").show()
con.sql(f"SELECT MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest FROM {TABLES['dim_clients']}").show()

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│   min_d    │   max_d    │  n_rows  │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │ 78835655 │
└────────────┴────────────┴──────────┘

┌──────────────┬────────────┬────────┐
│ is_published │ is_deleted │   n    │
│   boolean    │  boolean   │ int64  │
├──────────────┼────────────┼────────┤
│ true         │ false      │ 411540 │
│ false        │ true       │ 101559 │
│ false        │ false      │   6507 │
└──────────────┴────────────┴────────┘

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-02 │
└────────────┴────────────┘



### Label

**Target `decline_next_30d`**, scored at each month-end snapshot t.

- `last30` = the page's impressions in the 30 days up to and including t. `f1` and `f2` = its impressions in days 1 to 15 and days 16 to 30 after t.
- Page ratio = (f1 + f2) / last30. Client ratio = the same ratio summed over the client's eligible pages (like for like). `rel` = page ratio / client ratio.
- Label = 1 when `rel` is at most 0.5 and both halves are below baseline relative to the client: (f1 / last30) / (client f1 / client last30) < 1, and the same for f2. Otherwise 0.
- Not labeled: pages under 100 impressions in the 30 days before t, and clients with too few eligible pages for a stable benchmark (minimum set below).

**Why relative to the client.** A client-wide shift (seasonality, a tracking change) moves every page together. Comparing each page with its own client keeps those out of the label.

**Why like for like.** Benchmarking against all of a client's pages inflated the benchmark with new, ramping pages. At t = 2026-03-31 the median relative ratio was 0.772 against all pages and 0.837 against eligible pages (64,807 pages).

**Why a 0.5 cut.** Base rates with persistence at cut 0.8 were 25.0% (2026-01-31), 31.4% (2026-02-28) and 44.6% (2026-03-31), so a random queue would already score near the base rate. At cut 0.5 they were 6.7%, 10.2% and 19.9%. The cut was chosen by meaning and base rate on training snapshots, never by model performance, and the sealed test month was not touched.

**The base rate drifts.** The typical page's raw 30-day ratio fell from 1.26 to 1.081 to 0.756 across those three snapshots. Precision@K is therefore not comparable across months, and every result is reported with its base rate and lift (precision divided by base rate).

**Minimum client size.** A client needs at least 100 eligible pages at t to get a label, because a benchmark built from fewer pages is unstable. At t = 2026-03-31 this keeps 16 of 26 clients and 64,669 of 64,807 pages (99.8%).

**Concentration.** At that snapshot six clients with 2,000+ eligible pages hold 89.2% of eligible pages, and the three largest hold 59.3%. Pooled results mostly reflect these clients, so Results reports per-client breakdowns and a client-held-out check, and claims are limited to this client mix.

**Training set.** Twelve month-end snapshots (2025-04-30 to 2026-03-31) give 219,258 labeled rows and 22,429 positives (10.2%), after the 200-page client minimum and the exclusion of new or spiking pages, with up to 13 clients per snapshot. The base rate ranges from 3.2% to 21.4% by snapshot. The first six snapshots (Apr to Sep 2025) have only 1 to 5 clients each (35,090 rows, 16.0%); a sensitivity run without them is reported. The same page appears in many snapshots, so rows are not independent: validation is by time, with a client-held-out check as a robustness test.

### Features

Every feature uses data up to and including t; the label uses data after t only. Pages are the unit; all windows are 30 days (90 for the full feature window).

| Group | Features |
|---|---|
| Volume and engagement (90 days) | `imp_90`, `clicks_90`, `ctr_90`, `ctr_last30`, `active_days_90`, `cv_90` (volatility of daily impressions) |
| Position | `pos_90`, `pos_last30`, `pos_prev30`, `pos_change` (impression-weighted: sum of position / impressions) |
| Momentum | `imp_last30`, `imp_prev30`, `imp_prev60`, `log_mom`, `log_mom_prev`, and the client-relative versions `log_mom_rel`, `log_mom_prev_rel` (client benchmark over the same eligible pages) |
| Age | `page_age_days` at t |
| Static content fields (export-time snapshot) | word and character counts, content type, intent, competition, search volume, cpc, backlinks, category count, keyword tokens, URL length. Reported with and without; `backlinks` may carry information from after t. |

Not used in the main model: GA4 metrics (ablation only), client identity and client size, `last_optimized_date`, `fact_query_90d`.

In [3]:
import pandas as pd

def build_label_base(t_date, benchmark='universe', table='label_base'):
    """One row per eligible page at snapshot t, with the label ingredients.
    benchmark='universe': client ratio computed over the same eligible pages (like for like).
    benchmark='all': client ratio computed over every page of the client (the first version)."""
    t = pd.Timestamp(t_date)
    half = LABEL_DAYS // 2
    months = sorted({(t + pd.Timedelta(days=k)).strftime('%Y-%m') for k in (-(LABEL_DAYS - 1), 0, 1, LABEL_DAYS)})
    months_sql = ', '.join(f"'{m}'" for m in months)
    bench_src = 'elig' if benchmark == 'universe' else 'page'
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id, gsc_impressions
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {LABEL_DAYS}
              AND report_date <= DATE '{t_date}' + {LABEL_DAYS}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, SUM(last30) AS c_last30, SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM {bench_src}
            GROUP BY 1
        )
        SELECT e.client_hash_id, e.content_hash_id, e.last30, e.f1, e.f2,
               1.0 * (e.f1 + e.f2) / e.last30 AS raw,
               1.0 * (c.c_f1 + c.c_f2) / c.c_last30 AS c_ratio,
               (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
               (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
               (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2
        FROM elig e
        JOIN cl c ON c.client_hash_id = e.client_hash_id
        WHERE c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
    """)

def centering_summary(table, label):
    return con.sql(f"""
        SELECT '{label}' AS run, COUNT(*) AS n_pages,
               ROUND(quantile_cont(raw, 0.5), 3) AS median_raw_ratio,
               ROUND(quantile_cont(rel, 0.5), 3) AS median_rel,
               ROUND(AVG(CAST(rel <= 0.8 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_8,
               ROUND(AVG(CAST(rel <= 0.5 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_5
        FROM {table}
    """)

# D1. Same snapshot, two benchmarks: all pages of the client vs the eligible pages only
build_label_base('2026-03-31', benchmark='all', table='lb_all')
build_label_base('2026-03-31', benchmark='universe', table='lb_universe')
centering_summary('lb_all', 'Mar-31, benchmark = all pages').show()
centering_summary('lb_universe', 'Mar-31, benchmark = eligible pages').show()

# D2. Placebo: the same like-for-like definition at two other snapshots
for t_other in ['2026-02-28', '2026-01-31']:
    build_label_base(t_other, benchmark='universe', table='lb_other')
    centering_summary('lb_other', f'{t_other}, benchmark = eligible pages').show()

# D3. How big and how fast-moving are the client benchmarks? (client_ratio > 1 = client grew)
con.sql("""
    SELECT COUNT(*) AS n_clients,
           ROUND(quantile_cont(c_ratio, 0.5), 3) AS median_client_ratio,
           quantile_cont(n_pages, [0.1, 0.5, 0.9]) AS pages_per_client_q10_50_90
    FROM (SELECT client_hash_id, ANY_VALUE(c_ratio) AS c_ratio, COUNT(*) AS n_pages
          FROM lb_universe GROUP BY 1)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│              run              │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│            varchar            │  int64  │      double      │   double   │      double       │      double       │
├───────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ Mar-31, benchmark = all pages │   64809 │            0.756 │      0.772 │             0.507 │             0.229 │
└───────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                run                 │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│              varchar               │  int64  │      double 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-02-28, benchmark = eligible pages │   55525 │            1.081 │      0.963 │             0.314 │             0.102 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-01-31, benchmark = eligible pages │   41324 │             1.26 │       1.07 │              0.25 │             0.067 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌───────────┬─────────────────────┬────────────────────────────┐
│ n_clients │ median_client_ratio │ pages_per_client_q10_50_90 │
│   int64   │       double        │          double[]          │
├───────────┼─────────────────────┼───────────────────

In [4]:
# Client size and dominance at 2026-03-31 (like-for-like benchmark table from the previous cell)
con.sql("""
    WITH c AS (SELECT client_hash_id, COUNT(*) AS n_pages FROM lb_universe GROUP BY 1)
    SELECT m AS min_pages_per_client,
           COUNT(*) FILTER (WHERE n_pages >= m) AS clients_kept,
           SUM(n_pages) FILTER (WHERE n_pages >= m) AS pages_kept
    FROM c CROSS JOIN (VALUES (10), (30), (50), (100)) AS t(m)
    GROUP BY m
    ORDER BY m
""").show()

con.sql("""
    WITH c AS (SELECT client_hash_id, COUNT(*) AS n_pages FROM lb_universe GROUP BY 1),
         r AS (SELECT n_pages, ROW_NUMBER() OVER (ORDER BY n_pages DESC) AS rk FROM c)
    SELECT ROUND(1.0 * SUM(n_pages) FILTER (WHERE rk <= 1) / SUM(n_pages), 3) AS top1_client_share,
           ROUND(1.0 * SUM(n_pages) FILTER (WHERE rk <= 3) / SUM(n_pages), 3) AS top3_client_share
    FROM r
""").show()

con.sql("""
    WITH s AS (SELECT *, COUNT(*) OVER (PARTITION BY client_hash_id) AS n_pages FROM lb_universe)
    SELECT CASE WHEN n_pages < 30 THEN 'a: under 30'
                WHEN n_pages < 200 THEN 'b: 30-199'
                WHEN n_pages < 2000 THEN 'c: 200-1999'
                ELSE 'd: 2000+' END AS eligible_pages_in_client,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(*) AS n_pages,
           ROUND(AVG(CAST(rel <= 0.5 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_5
    FROM s
    GROUP BY 1
    ORDER BY 1
""").show()

┌──────────────────────┬──────────────┬────────────┐
│ min_pages_per_client │ clients_kept │ pages_kept │
│        int32         │    int64     │   int128   │
├──────────────────────┼──────────────┼────────────┤
│                   10 │           22 │      64794 │
│                   30 │           18 │      64741 │
│                   50 │           16 │      64669 │
│                  100 │           16 │      64669 │
└──────────────────────┴──────────────┴────────────┘

┌───────────────────┬───────────────────┐
│ top1_client_share │ top3_client_share │
│      double       │      double       │
├───────────────────┼───────────────────┤
│             0.261 │             0.593 │
└───────────────────┴───────────────────┘

┌──────────────────────────┬───────────┬─────────┬───────────────────┐
│ eligible_pages_in_client │ n_clients │ n_pages │ base_rate_cut_0_5 │
│         varchar          │   int64   │  int64  │      double       │
├──────────────────────────┼───────────┼─────────┼──────

In [5]:
REL_MAX = 0.5            # locked: relative ratio at most 0.5
MIN_CLIENT_PAGES = 100   # a client needs this many eligible pages at t for a stable benchmark

def build_labels(t_date, table='labels'):
    """Final label for snapshot t. One row per eligible page; y = decline_next_30d."""
    t = pd.Timestamp(t_date)
    half = LABEL_DAYS // 2
    months = sorted({(t + pd.Timedelta(days=k)).strftime('%Y-%m') for k in (-(LABEL_DAYS - 1), 0, 1, LABEL_DAYS)})
    months_sql = ', '.join(f"'{m}'" for m in months)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id, gsc_impressions
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {LABEL_DAYS}
              AND report_date <= DATE '{t_date}' + {LABEL_DAYS}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, SUM(last30) AS c_last30, SUM(f1) AS c_f1, SUM(f2) AS c_f2,
                   COUNT(*) AS n_elig
            FROM elig
            GROUP BY 1
        ),
        scored AS (
            SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t,
                   e.last30, e.f1, e.f2, c.n_elig,
                   (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
                   (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
                   (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2
            FROM elig e
            JOIN cl c ON c.client_hash_id = e.client_hash_id
            WHERE c.n_elig >= {MIN_CLIENT_PAGES}
              AND c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
        )
        SELECT *, CAST(rel <= {REL_MAX} AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER) AS y
        FROM scored
    """)

# Base rate per training snapshot (the sealed test snapshot is not touched here)
rows = []
for t_str in SNAPSHOTS['train']:
    build_labels(t_str, table='lb_tmp')
    n, k, pos, br = con.sql("""
        SELECT COUNT(*), COUNT(DISTINCT client_hash_id), COALESCE(SUM(y), 0), ROUND(AVG(y), 3)
        FROM lb_tmp
    """).fetchone()
    rows.append((t_str, n, k, pos, br))
label_summary = pd.DataFrame(rows, columns=['snapshot_t', 'n_pages', 'n_clients', 'n_pos', 'base_rate'])
label_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,snapshot_t,n_pages,n_clients,n_pos,base_rate
0,2025-04-30,1482,1,51,0.034
1,2025-05-31,3988,2,632,0.158
2,2025-06-30,7390,4,891,0.121
3,2025-07-31,8172,4,788,0.096
4,2025-08-31,9024,4,2082,0.231
5,2025-09-30,9755,6,2086,0.214
6,2025-10-31,16036,7,968,0.060
7,2025-11-30,21827,8,1203,0.055
8,2025-12-31,33197,11,3162,0.095
9,2026-01-31,41218,14,2760,0.067


In [7]:
# Is missingness a client fingerprint? One row per client, ranked by size (ids not shown)
con.sql("""
    WITH s AS (
        SELECT client_hash_id, COUNT(*) AS n,
               AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END) AS miss_word_count,
               AVG(CASE WHEN backlinks IS NULL THEN 1.0 ELSE 0 END) AS miss_backlinks,
               AVG(CASE WHEN competition IS NULL THEN 1.0 ELSE 0 END) AS miss_keyword_fields,
               AVG(y) AS base_rate
        FROM train_set
        GROUP BY 1
    )
    SELECT ROW_NUMBER() OVER (ORDER BY n DESC) AS client_rank, n,
           ROUND(miss_word_count, 2) AS miss_word_count,
           ROUND(miss_backlinks, 2) AS miss_backlinks,
           ROUND(miss_keyword_fields, 2) AS miss_keyword_fields,
           ROUND(base_rate, 3) AS base_rate
    FROM s
    ORDER BY client_rank
""").show(max_rows=30)

# Does missing mean something for the outcome?
con.sql("""
    SELECT word_count IS NULL AS word_count_missing, backlinks IS NULL AS backlinks_missing,
           COUNT(*) AS n, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

CatalogException: Catalog Error: Table with name train_set does not exist!
Did you mean "pg_attribute"?

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

K_LIST = [20, 50]
d = con.sql("""
    SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30
    FROM train_set
""").df()

# Rule baselines: a higher score means "look at this page first"
d['rule_rel_momentum'] = -d['log_mom_rel']                      # fell most, relative to its client
d['rule_raw_momentum'] = -d['log_mom']                          # fell most, no client adjustment
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']  # lost the most impressions in absolute terms
RULES = ['rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

# Summary: mean over the 12 training snapshots. A random queue has lift 1.0 by construction.
summary = []
for r in RULES:
    e = eval_rule(d, r)
    summary.append({'rule': r, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
print(pd.DataFrame(summary).to_string(index=False))

# Per-snapshot detail for the main rule
eval_rule(d, 'rule_rel_momentum')

In [ ]:
import os

assert FEATURE_DAYS == 3 * LABEL_DAYS, 'trend windows assume 90 = 3 x 30'

PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']

def build_snapshot(t_date, table='snap'):
    """Labels (same rules as build_labels) plus past-only features for one snapshot t.
    Features use report_date <= t only; the label uses report_date > t only."""
    t = pd.Timestamp(t_date)
    L = LABEL_DAYS
    half = L // 2
    days = pd.date_range(t - pd.Timedelta(days=FEATURE_DAYS - 1), t + pd.Timedelta(days=L))
    months_sql = ', '.join(f"'{m}'" for m in sorted({d.strftime('%Y-%m') for d in days}))
    static_sql = ', '.join(f"d.{c}" for c in STATIC_COLS)
    static_out = ', '.join(f"e.{c}" for c in STATIC_COLS)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id,
                   gsc_impressions, gsc_clicks, gsc_sum_position
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {FEATURE_DAYS}
              AND report_date <= DATE '{t_date}' + {L}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   -- label side: strictly after t
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2,
                   -- feature side: up to and including t
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS imp_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_clicks ELSE 0 END) AS clicks_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_sum_position ELSE 0 END) AS pos_sum_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions * gsc_impressions ELSE 0 END) AS imp_sq_90,
                   COUNT(*) FILTER (WHERE report_date <= DATE '{t_date}' AND gsc_impressions > 0) AS active_days_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_clicks ELSE 0 END) AS clicks_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_impressions ELSE 0 END) AS prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {2*L} THEN gsc_impressions ELSE 0 END) AS prev60
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*, d.content_created_date, {static_sql}
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, COUNT(*) AS n_elig,
                   SUM(last30) AS c_last30, SUM(prev30) AS c_prev30, SUM(prev60) AS c_prev60,
                   SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM elig
            GROUP BY 1
        ),
        scored AS (
            SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t, c.n_elig,
                   -- label ingredients
                   (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
                   (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
                   (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2,
                   -- past-only features
                   e.imp_90, e.clicks_90,
                   1.0 * e.clicks_90 / NULLIF(e.imp_90, 0) AS ctr_90,
                   1.0 * e.clicks_last30 / NULLIF(e.last30, 0) AS ctr_last30,
                   1.0 * e.pos_sum_90 / NULLIF(e.imp_90, 0) AS pos_90,
                   1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0) AS pos_last30,
                   1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0) AS pos_prev30,
                   (1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0))
                     - (1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0)) AS pos_change,
                   e.last30 AS imp_last30, e.prev30 AS imp_prev30, e.prev60 AS imp_prev60,
                   e.active_days_90,
                   SQRT(GREATEST(1.0 * e.imp_sq_90 / {FEATURE_DAYS} - POWER(1.0 * e.imp_90 / {FEATURE_DAYS}, 2), 0))
                     / NULLIF(1.0 * e.imp_90 / {FEATURE_DAYS}, 0) AS cv_90,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0)) AS log_mom,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 THEN LN(1.0 * c.c_last30 / c.c_prev30) END AS log_mom_rel,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0)) AS log_mom_prev,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 AND c.c_prev60 > 0 THEN LN(1.0 * c.c_prev30 / c.c_prev60) END AS log_mom_prev_rel,
                   DATE '{t_date}' - e.content_created_date AS page_age_days,
                   {static_out}
            FROM elig e
            JOIN cl c ON c.client_hash_id = e.client_hash_id
            WHERE c.n_elig >= {MIN_CLIENT_PAGES}
              AND c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
        )
        SELECT *, CAST(rel <= {REL_MAX} AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER) AS y
        FROM scored
    """)

CACHE_DIR = 'snap_cache'   # local to this Colab runtime; never commit it (no raw exports in the repo)
os.makedirs(CACHE_DIR, exist_ok=True)
expected = label_summary.set_index('snapshot_t')
for t_str in SNAPSHOTS['train']:
    path = f'{CACHE_DIR}/snap_{t_str}.parquet'
    if os.path.exists(path):
        continue                      # already built in this runtime
    build_snapshot(t_str, table='snap_tmp')
    n, pos = con.sql("SELECT COUNT(*), COALESCE(SUM(y), 0) FROM snap_tmp").fetchone()
    # the combined build must reproduce the locked label exactly
    assert (n, pos) == (expected.loc[t_str, 'n_pages'], expected.loc[t_str, 'n_pos']), f'label mismatch at {t_str}'
    con.sql(f"COPY snap_tmp TO '{path}' (FORMAT PARQUET)")
con.sql(f"CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM read_parquet('{CACHE_DIR}/snap_*.parquet')")

con.sql("SELECT COUNT(*) AS n_rows, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate FROM train_set").show()

# Missing values per feature (decides imputation later)
all_feats = PAST_COLS + STATIC_COLS
nulls = con.sql("SELECT " + ", ".join(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in all_feats)
                + " FROM train_set").df().T
nulls.columns = ['n_missing']
nulls['share_missing'] = (nulls['n_missing'] / con.sql("SELECT COUNT(*) FROM train_set").fetchone()[0]).round(4)
nulls

In [ ]:
MIN_CLIENT_PAGES_MODEL = 200   # replaces 100; filtering on n_elig is equivalent to rebuilding with MIN_CLIENT_PAGES = 200

# Unfiltered table, rebuilt from the cached snapshots (safe to re-run)
con.sql(f"CREATE OR REPLACE TEMP TABLE train_set_all AS SELECT * FROM read_parquet('{CACHE_DIR}/snap_*.parquet')")

# What does the 200 rule remove? (a client can move between buckets as it grows)
con.sql("""
    SELECT CASE WHEN n_elig < 200 THEN '100-199' WHEN n_elig < 500 THEN '200-499'
                WHEN n_elig < 2000 THEN '500-1999' ELSE '2000+' END AS client_size_at_t,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(*) AS n_rows, ROUND(AVG(y), 3) AS base_rate
    FROM train_set_all
    GROUP BY 1
    ORDER BY MIN(n_elig)
""").show()

con.sql(f"CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM train_set_all WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}")
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate FROM train_set").show()

# Does the label rate depend on past relative momentum? decile 1 = fell most, 10 = rose most (deciles within each snapshot)
for scope, cond in [('all 12 snapshots', "TRUE"), ('last 4 snapshots', "snap_t >= DATE '2025-12-31'")]:
    print(scope)
    con.sql(f"""
        SELECT decile, COUNT(*) AS n, ROUND(AVG(y), 3) AS label_rate, ROUND(AVG(log_mom_rel), 2) AS avg_log_mom_rel
        FROM (SELECT y, log_mom_rel,
                     NTILE(10) OVER (PARTITION BY snap_t ORDER BY log_mom_rel) AS decile
              FROM train_set WHERE {cond})
        GROUP BY decile
        ORDER BY decile
    """).show()

# Baselines again on the filtered set
d = con.sql("SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30 FROM train_set").df()
d['rule_rel_momentum'] = -d['log_mom_rel']
d['rule_raw_momentum'] = -d['log_mom']
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']

def summarize(df, label):
    out = []
    for r in RULES:
        e = eval_rule(df, r)
        out.append({'rule': r, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
    print(label)
    print(pd.DataFrame(out).to_string(index=False))

summarize(d, 'Baselines, all 12 training snapshots')
summarize(d[d['snap_t'].astype(str) >= '2025-12-31'], 'Baselines, last 4 snapshots (Dec 2025 to Mar 2026)')

In [ ]:
# Per-snapshot table after the 200-page client minimum
con.sql("""
    SELECT snap_t, COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
           SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1
    ORDER BY 1
""").show(max_rows=20)

# How much of the label is "a recent spike falling back"? (descriptive state at t, past data only; thresholds are exploratory)
for scope, cond in [('all 12 snapshots', "TRUE"), ('last 4 snapshots', "snap_t >= DATE '2025-12-31'")]:
    print(scope)
    con.sql(f"""
        WITH s AS (
            SELECT y,
                   CASE WHEN imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60) THEN 'a: spike (last 30d >= 2x both earlier months)'
                        WHEN imp_last30 <= 0.8 * imp_prev30 AND imp_prev30 <= 0.8 * imp_prev60 THEN 'b: steady decline (two falls of 20%+)'
                        ELSE 'c: other' END AS state_at_t
            FROM train_set
            WHERE {cond}
        )
        SELECT state_at_t, COUNT(*) AS n,
               ROUND(1.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share_of_rows,
               ROUND(AVG(y), 3) AS label_rate,
               ROUND(1.0 * SUM(y) / SUM(SUM(y)) OVER (), 3) AS share_of_positives
        FROM s
        GROUP BY 1
        ORDER BY 1
    """).show()

In [ ]:
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"   # new or spiking at t (past data only)

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE train_set_flagged AS
    SELECT *, ({SPIKE_RULE}) AS is_spike
    FROM train_set_all
    WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}
""")
con.sql("CREATE OR REPLACE TEMP TABLE train_spike AS SELECT * FROM train_set_flagged WHERE is_spike")
con.sql("CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM train_set_flagged WHERE NOT is_spike")

# What leaves the decline model, and what stays
con.sql("""
    SELECT is_spike, COUNT(*) AS n_rows, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set_flagged
    GROUP BY 1
    ORDER BY 1
""").show()

con.sql("""
    SELECT snap_t, COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
           SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1
    ORDER BY 1
""").show(max_rows=20)

# Label rate by past relative momentum on the remaining pages (decile 1 = fell most)
con.sql("""
    SELECT decile, COUNT(*) AS n, ROUND(AVG(y), 3) AS label_rate, ROUND(AVG(log_mom_rel), 2) AS avg_log_mom_rel
    FROM (SELECT y, log_mom_rel,
                 NTILE(10) OVER (PARTITION BY snap_t ORDER BY log_mom_rel) AS decile
          FROM train_set)
    GROUP BY decile
    ORDER BY decile
""").show()

# Baselines again on the non-spike pages
d = con.sql("SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30 FROM train_set").df()
d['rule_rel_momentum'] = -d['log_mom_rel']
d['rule_raw_momentum'] = -d['log_mom']
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']
summarize(d, 'Baselines on non-spike pages, all 12 training snapshots')
summarize(d[d['snap_t'].astype(str) >= '2025-12-31'], 'Baselines on non-spike pages, last 4 snapshots')

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

LOG_COLS = ['imp_90', 'clicks_90', 'imp_last30', 'imp_prev30', 'imp_prev60']   # count features, logged for the linear model

data = con.sql(f"SELECT snap_t, y, {', '.join(PAST_COLS)} FROM train_set").df()
data['snap_t'] = pd.to_datetime(data['snap_t'])

def make_X(df, log=False):
    X = df[PAST_COLS].copy()
    for c in X.columns:
        if pd.api.types.is_timedelta64_dtype(X[c]):
            X[c] = X[c].dt.days          # in case a date difference arrives as an interval
    X = X.astype(float)
    if log:
        for c in LOG_COLS:
            X[c] = np.log1p(X[c])
    return X

def make_hgb(seed=0):
    # hyperparameters fixed in advance, not tuned on any evaluation snapshot
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=200, l2_regularization=1.0,
                                          early_stopping=False, random_state=seed)

def fit_score(train, test, which, seed=0):
    y_tr = train['y'].values
    if which == 'hgb':
        m = make_hgb(seed).fit(make_X(train), y_tr)
        return m.predict_proba(make_X(test))[:, 1]
    if which == 'logit':
        m = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
                          LogisticRegression(C=0.1, max_iter=1000))
        m.fit(make_X(train, log=True), y_tr)
        return m.predict_proba(make_X(test, log=True))[:, 1]
    if which == 'hgb_shuffled':   # control: same model, labels shuffled inside the training fold
        y_sh = np.random.default_rng(seed).permutation(y_tr)
        m = make_hgb(seed).fit(make_X(train), y_sh)
        return m.predict_proba(make_X(test))[:, 1]

# Walk-forward on the last 6 training snapshots. Embargo: a training snapshot's label window
# (30 days) must end on or before the evaluation snapshot date.
EVAL_SNAPS = SNAPSHOTS['train'][-6:]
parts = []
for t_eval in EVAL_SNAPS:
    te_t = pd.Timestamp(t_eval)
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    test = data[data['snap_t'] == te_t].copy()
    if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
        print(f"{t_eval}: skipped (no usable training rows)")
        continue
    print(f"{t_eval}: train rows {len(train):,} (up to {train['snap_t'].max().date()}), test rows {len(test):,}")
    for which in ['hgb', 'logit', 'hgb_shuffled']:
        test[which] = fit_score(train, test, which)
    test['rule_rel_momentum'] = -test['log_mom_rel']
    test['rule_raw_momentum'] = -test['log_mom']
    test['rule_impressions_lost'] = test['imp_prev30'] - test['imp_last30']
    parts.append(test)
res = pd.concat(parts)

SCORES = ['hgb', 'logit', 'hgb_shuffled', 'rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']
summary = []
for s in SCORES:
    e = eval_rule(res, s)
    summary.append({'score': s, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
print()
print(pd.DataFrame(summary).to_string(index=False))

# Per-snapshot detail for the main model
eval_rule(res, 'hgb')

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
